# Vector Stores

**LangChain 1.0.5+ | Mixed Level Class**

## 🎯 Objectives
1. Understand vector stores
2. Use InMemoryVectorStore (testing)
3. Use FAISS (production)
4. Use Chroma (persistent)
5. Compare vector stores

In [1]:
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()
print("✅ Setup complete")

✅ Setup complete


## 1. What are Vector Stores? 🤔

### 🔰 BEGINNER

**Vector Stores** are databases optimized for finding similar vectors.

Like a library:
- Traditional database: "Find book with ISBN 12345"
- Vector store: "Find books similar to this topic"

They enable **semantic search**!

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# Initialize
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)

# Add documents
docs = [
    Document(page_content="Python is a programming language"),
    Document(page_content="Machine learning uses algorithms"),
    Document(page_content="The sun is a star")
]

vector_store.add_documents(docs)
print(f"✅ Added {len(docs)} documents\n")

# Search
results = vector_store.similarity_search("What is Python?", k=2)

print("Search Results:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")

✅ Added 3 documents

Search Results:
1. Python is a programming language
2. Machine learning uses algorithms


## 3. FAISS (Fast & Production-Ready)

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

# Load and split a PDF
pdf_path = "pdfs/rag.pdf"

if Path(pdf_path).exists():
    print("Loading PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Split
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    
    # Create FAISS vectorstore
    print("Creating embeddings (this may take a minute)...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    
    # Save to disk
    vectorstore.save_local("./faiss_index_notebook")
    print("✅ FAISS index saved\n")
    
    # Search
    query = "What is RAG?"
    results = vectorstore.similarity_search(query, k=3)
    
    print(f"Query: {query}\n")
    for i, doc in enumerate(results, 1):
        print(f"Result {i}:")
        print(f"  {doc.page_content[:150]}...\n")
else:
    print(f"❌ PDF not found: {pdf_path}")

Loading PDF...
Split into 92 chunks
Creating embeddings (this may take a minute)...
✅ FAISS index saved

Query: What is RAG?

Result 1:
  in 71% of cases, and a gold article is present in the top 10 retrieved articles in 90% of cases.
4.5 Additional Results
Generation Diversity Section 4...

Result 2:
  2 Methods
We explore RAG models, which use the input sequencex to retrieve text documentsz and use them
as additional context when generating the targ...

Result 3:
  blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found
at https://huggingface.co/rag/
2https://github.com/pytorch/fair...



In [4]:
query = "What is RAG-Sequence Model?"
results = vectorstore.similarity_search(query, k=3)
results

[Document(id='0c9ad125-6e86-4460-9627-c07ae89a1211', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2'}, page_content='2 Methods\nWe explore RAG models, which use the input sequencex to retrieve text documentsz and use them\nas additional context when generating the target sequence y. As shown in Figure 1, our models\nleverage two components: (i) a retriever pη(z|x) with parametersη that returns (top-K truncated)\ndistributions over text passages given a queryx and (ii) a generatorpθ(yi|x,z,y 1:i−1) parametrized\n1Code to run experiments with RAG has been open-sourced as part of the HuggingFace Transform-\ners Library [66]

In [5]:
# Load existing index
if Path("./faiss_index_notebook").exists():
    loaded_vectorstore = FAISS.load_local(
        "./faiss_index_notebook",
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Loaded existing FAISS index")
    
    # Use it
    results = loaded_vectorstore.similarity_search("transformers", k=2)
    print(f"\nFound {len(results)} results")
else:
    print("No existing index found")

✅ Loaded existing FAISS index

Found 2 results


## 5. Chroma (Persistent & Easy)

In [6]:
from langchain_chroma import Chroma

# Create Chroma vectorstore
chroma_store = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_notebook"
)

# Add documents
sample_docs = [
    Document(
        page_content="RAG combines retrieval and generation",
        metadata={"topic": "rag", "difficulty": "intermediate"}
    ),
    Document(
        page_content="LangChain simplifies LLM applications",
        metadata={"topic": "langchain", "difficulty": "beginner"}
    )
]

chroma_store.add_documents(sample_docs)
print("✅ Added documents to Chroma\n")

# Search with metadata filter
results = chroma_store.similarity_search(
    "Tell me about RAG",
    k=2,
    filter={"topic": "rag"}
)

print("Filtered search results:")
for doc in results:
    print(f"  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")

✅ Added documents to Chroma

Filtered search results:
  RAG combines retrieval and generation
  Metadata: {'difficulty': 'intermediate', 'topic': 'rag'}


## 6. Vector Store Comparison

| Store | Persistent | Speed | Metadata Filtering | Best For |
|-------|-----------|-------|-------------------|----------|
| **InMemory** | ❌ | ⚡⚡⚡ | ❌ | Testing |
| **FAISS** | ✅ | ⚡⚡⚡ | Limited | Production, Speed |
| **Chroma** | ✅ | ⚡⚡ | ✅ | Development, Filtering |

## Summary

✅ Vector stores enable semantic search
✅ InMemory for testing
✅ FAISS for speed and scale
✅ Chroma for metadata filtering
✅ Always persist for production!

**Next:** Retrieval Strategies!